# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import subprocess, sys

# Install required libraries
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)

import os
from google.colab import userdata

# Get the Hugging Face token from Colab Secrets
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print("Setup done. Token loaded:", "Yes" if os.environ.get("HF_TOKEN") else "No")

Setup done. Token loaded: Yes


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content page's daily performance data (for one client, on
one specific date), from the fact_content_daily_performance table.

Time window: I will use March 2026 (month=2026-03) to build and test my
approach, and keep the final month (June 2026) sealed for later testing.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*


## 2. Fields: feature / label / context / excluded

**Feature:** These are the inputs used to calculate the recommendation/score —
gsc_impressions, gsc_clicks, gsc_avg_position, ga4_pageviews, ga4_sessions,
ga4_engaged_sessions, scroll_events, and the traffic-source breakdown
(sessions_organic, sessions_direct, sessions_referral, sessions_social,
sessions_paid).

**Label:** There is no ready-made label in this table, so I will build a
proxy myself using gsc_impressions — comparing an earlier window vs. a
later window to detect decline.

**Context:** report_date, client_hash_id, content_hash_id, client_has_gsc,
client_has_ga4, gsc_data_available, ga4_data_available — used for joining,
grouping, time windows, and availability checks, not fed to the model as
features.

**Excluded:** Nothing needed to be excluded from this table, since FlyRank
already removed product decision fields (like health_score or priority_score)
before releasing this dataset. If such a field ever existed in my data, I
would exclude it, because it would encode FlyRank's own past decision and
leak the answer into the model instead of letting the model discover its
own signal.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
#since one row means one page at a date so if duplicate comes it means our claim was wrong if not then it means our claim was right and hence proved!
import duckdb

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

# Grain check: is one row really "one page, one client, one date"?
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as row_count
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

print("Rows where grain is violated (should be EMPTY if grain is correct):")
print(grain_check)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows where grain is violated (should be EMPTY if grain is correct):
Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, row_count]
Index: []


**Grain verification result:** The query returned an empty DataFrame —
no (client, content, date) combination appears more than once. This
confirms my Section 1 claim: one row truly represents one content page,
for one client, on one specific date.

In [4]:
#now check count of march data ad give me a number
count_and_dates = con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        MIN(report_date) as earliest_date,
        MAX(report_date) as latest_date
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print(count_and_dates)


   total_rows earliest_date latest_date
0     9841378    2026-03-01  2026-03-31


we have been given that the now of rows of march data is 9841378

In [5]:
#now let us check which row is empty and how many are complete

availability_check = con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) as rows_with_gsc,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) as rows_with_ga4,
        ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) / COUNT(*), 1) as pct_gsc_available,
        ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 1) as pct_ga4_available
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  rows_with_gsc  rows_with_ga4  pct_gsc_available  \
0     9841378        3611061         413966               36.7   

   pct_ga4_available  
0                4.2  


**Availability check result:** Out of 9,841,378 rows in March 2026, only
36.7% have valid GSC data and only 4.2% have valid GA4 data. This confirms
the need to filter with `gsc_data_available IS TRUE` (and `ga4_data_available
IS TRUE` when using GA4 signals) before building features — otherwise,
missing/untracked rows would be mistaken for pages with zero traffic.

In [6]:
# now for windows proof
window_check = con.sql(f"""
    SELECT
        MIN(report_date) as earliest_date,
        MAX(report_date) as latest_date,
        COUNT(DISTINCT report_date) as unique_dates
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

print(window_check)

  earliest_date latest_date  unique_dates
0    2026-03-01  2026-03-31            31


Window claim is also proved which shows that my data is only for march month not for the june month.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## 4. Data limits

**What this data can never tell me:**

1. **Real identities** — All client names, page URLs, and search queries
   are scrambled (hashed) before I ever see them. I can never know which
   real website or real search term a row belongs to — only pseudonymized
   IDs. This is intentional (privacy protection), but it means I can never
   name or investigate a specific real page.

2. **Unbalanced history** — Different clients started being tracked at
   different times. Some have 12+ months of history, some only a couple
   months. I can never fairly compare "early" vs "late" trends across all
   clients equally, because they don't all have the same amount of history.

3. **GSC-only early rows** — For some clients, GA4 (analytics) tracking
   started later than GSC (search) tracking. In their early rows, I only
   have search data, not analytics data. This data can never tell me about
   user engagement/sessions for those early periods.

4. **No proof of "why"** — Even with clean data, this can never tell me
   *why* a page's traffic changed, or prove that refreshing a page *caused*
   it to recover. It only shows patterns, not causes.

5. **Window overlap risk** — If I'm not careful, my feature window and
   label window could accidentally overlap, leaking future information
   into my features. The data itself doesn't protect me from this — I
   have to build the separation myself.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.